# Experiment 9 — CIFAR-10 SHViT LENS Recovery Analysis

This notebook analyzes the seed-42 checkpoints from experiments 1, 4, 7, and 8 without training or modifying them.

- **Relative L2:** compares every shape-compatible parameter shared by the pretrained baseline and the recovered full-LENS model. Newly introduced LENS parameters and replaced LayerNorm parameters are excluded automatically.
- **Function curves:** plots 256 randomly selected test-set token/query curves together for every LayerNorm/Softmax module and its corresponding LENSNorm/LENSMax module.
- **Training histories:** compares each experiment's complete trajectory on a normalized 0–100% training-progress axis. Training curves start after the first epoch; validation curves include an explicitly evaluated 0% initialization point.

The structural replacement initially destroys the baseline behavior. Therefore, the relative-L2 results measure how the inherited parameters move during recovery after the architecture is changed; they are not ordinary additional fine-tuning statistics.

All outputs are written below `analysis/` using separate category and module directories.


In [1]:
import csv
import gc
import json
import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.ticker import LogFormatterSciNotation
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms


SEED = 42
PROJECT_ROOT = Path(".")
DATA_ROOT = PROJECT_ROOT / "data"
RUNS_ROOT = PROJECT_ROOT / "runs"
ANALYSIS_ROOT = PROJECT_ROOT / "analysis"

BASELINE_DIR = RUNS_ROOT / "1_cifar10_shvit_baseline_seed42"
SCRATCH_LENS_DIR = RUNS_ROOT / "4_cifar10_shvit_lens_full_from_scratch_seed42"
RECOVERED_LENS_DIR = RUNS_ROOT / "7_cifar10_shvit_lens_full_from_pretrained_seed42"
FROZEN_LENS_DIR = RUNS_ROOT / "8_cifar10_shvit_lens_full_from_pretrained_frozen_baseline_seed42"

BASELINE_BEST = BASELINE_DIR / "best.pt"
RECOVERED_LENS_BEST = RECOVERED_LENS_DIR / "best.pt"

IMAGE_SIZE = 32
PATCH_SIZE = 4
NUM_CLASSES = 10
EMBED_DIM = 128
DEPTH = 3
MLP_RATIO = 4.0
DROPOUT = 0.1
ATTENTION_DROPOUT = 0.0

LENS_NORM_HIDDEN = 32
LENS_MAX_ROW_HIDDEN = 32
LENS_MAX_ELEMENT_HIDDEN = 16

BATCH_SIZE = 256
NUM_WORKERS = 4
VALIDATION_SIZE = 5000
CURVE_BATCH_SIZE = 32
CURVES_PER_MODULE = 256
EPS = 1e-12

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

CATEGORY_DIRS = {
    "relative_weights": ANALYSIS_ROOT / "Relative_L2" / "weights",
    "relative_biases": ANALYSIS_ROOT / "Relative_L2" / "biases",
    "layernorm": ANALYSIS_ROOT / "LayerNorm",
    "softmax": ANALYSIS_ROOT / "Softmax",
    "lensnorm": ANALYSIS_ROOT / "LENSNorm",
    "lensmax": ANALYSIS_ROOT / "LENSMax",
    "history": ANALYSIS_ROOT / "Training_History",
}
for directory in CATEGORY_DIRS.values():
    directory.mkdir(parents=True, exist_ok=True)

required_files = [
    BASELINE_BEST,
    RECOVERED_LENS_BEST,
    BASELINE_DIR / "history.json",
    SCRATCH_LENS_DIR / "history.json",
    RECOVERED_LENS_DIR / "history.json",
    FROZEN_LENS_DIR / "history.json",
]
missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))

print(f"device={DEVICE}")
print(f"analysis_root={ANALYSIS_ROOT.resolve()}")


/home/yuhao/anaconda3/envs/brevitas/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device=cuda:0
analysis_root=/home/yuhao/project/LENS/SHViT/CIFAR10/analysis


In [2]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


class LENSNorm(nn.Module):
    """Predicts one affine scale and bias for every token."""

    def __init__(self, dim, hidden_dim=32, scale_min=0.05, scale_max=12.0):
        super().__init__()
        self.scale_min = scale_min
        self.scale_max = scale_max
        self.predictor_0 = nn.Linear(dim, hidden_dim)
        self.predictor_1 = nn.Linear(hidden_dim, 2)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.kaiming_normal_(self.predictor_0.weight, nonlinearity="relu")
        nn.init.zeros_(self.predictor_0.bias)
        nn.init.normal_(self.predictor_1.weight, mean=0.0, std=1e-3)
        with torch.no_grad():
            self.predictor_1.bias.copy_(torch.tensor([1.0, 0.0]))

    def forward(self, x):
        affine = self.predictor_1(F.relu(self.predictor_0(x)))
        scale = affine[..., 0:1].clamp(self.scale_min, self.scale_max)
        bias = affine[..., 1:2]
        return scale * x + bias


class LENSMax(nn.Module):
    """Maps each complete attention-score row to positive attention weights."""

    def __init__(self, num_tokens, row_hidden=32, element_hidden=16):
        super().__init__()
        self.num_tokens = num_tokens
        self.row_mlp = nn.Sequential(
            nn.Linear(num_tokens, row_hidden),
            nn.ReLU(),
            nn.Linear(row_hidden, 1),
        )
        self.element_mlp = nn.Sequential(
            nn.Linear(2, element_hidden),
            nn.ReLU(),
            nn.Linear(element_hidden, 1),
        )
        self.reset_parameters()

    def reset_parameters(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.kaiming_normal_(module.weight, nonlinearity="relu")
                nn.init.zeros_(module.bias)

    def forward(self, scores):
        batch_size, num_queries, num_keys = scores.shape
        if num_keys != self.num_tokens:
            raise ValueError(f"LENSMax expects {self.num_tokens} keys, received {num_keys}")

        flat_rows = scores.reshape(batch_size * num_queries, num_keys)
        row_state = self.row_mlp(flat_rows)
        row_state = row_state.unsqueeze(1).expand(-1, num_keys, -1)
        element_input = torch.cat((flat_rows.unsqueeze(-1), row_state), dim=-1)
        logits = self.element_mlp(element_input).squeeze(-1)

        divisor = float(2 ** int(math.floor(math.log2(num_keys))))
        weights = torch.sigmoid(logits) / divisor
        return weights.reshape(batch_size, num_queries, num_keys)


In [3]:
class SingleHeadSelfAttention(nn.Module):
    def __init__(self, dim, num_tokens, qkv_bias=True, attn_drop=0.0, proj_drop=0.0):
        super().__init__()
        self.scale = dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)
        self.lens_max = None

    def forward(self, x):
        batch_size, num_tokens, dim = x.shape
        qkv = self.qkv(x).reshape(batch_size, num_tokens, 3, dim).permute(2, 0, 1, 3)
        q, k, v = qkv.unbind(0)
        scores = (q @ k.transpose(-2, -1)) * self.scale

        if self.lens_max is None:
            attention = F.softmax(scores, dim=-1)
        else:
            attention = self.lens_max(scores)

        attention = self.attn_drop(attention)
        x = attention @ v
        x = self.proj(x)
        return self.proj_drop(x)


class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim, drop=0.0):
        super().__init__()
        self.fc1 = nn.Linear(dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, dim)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.drop(self.act(self.fc1(x)))
        return self.drop(self.fc2(x))


class TransformerBlock(nn.Module):
    def __init__(self, dim, num_tokens, mlp_ratio=4.0, drop=0.0, attn_drop=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = SingleHeadSelfAttention(
            dim=dim,
            num_tokens=num_tokens,
            attn_drop=attn_drop,
            proj_drop=drop,
        )
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = FeedForward(dim, int(dim * mlp_ratio), drop=drop)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class CIFAR10SHViT(nn.Module):
    def __init__(
        self,
        image_size=32,
        patch_size=4,
        num_classes=10,
        dim=128,
        depth=3,
        mlp_ratio=4.0,
        drop=0.1,
        attn_drop=0.0,
    ):
        super().__init__()
        self.dim = dim
        self.num_patches = (image_size // patch_size) ** 2
        self.num_tokens = self.num_patches + 1
        self.patch_embed = nn.Conv2d(3, dim, kernel_size=patch_size, stride=patch_size)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_tokens, dim))
        self.pos_drop = nn.Dropout(drop)
        self.blocks = nn.ModuleList([
            TransformerBlock(dim, self.num_tokens, mlp_ratio, drop, attn_drop)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, num_classes)
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.trunc_normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Conv2d):
            nn.init.kaiming_normal_(module.weight, mode="fan_out")
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)

    def forward(self, x):
        x = self.patch_embed(x).flatten(2).transpose(1, 2)
        cls_token = self.cls_token.expand(x.size(0), -1, -1)
        x = torch.cat((cls_token, x), dim=1)
        x = self.pos_drop(x + self.pos_embed)
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        return self.head(x[:, 0])


def install_lens_norm(model):
    for block in model.blocks:
        block.norm1 = LENSNorm(model.dim, hidden_dim=LENS_NORM_HIDDEN)
        block.norm2 = LENSNorm(model.dim, hidden_dim=LENS_NORM_HIDDEN)
    model.norm = LENSNorm(model.dim, hidden_dim=LENS_NORM_HIDDEN)


def install_lens_max(model):
    for block in model.blocks:
        block.attn.lens_max = LENSMax(
            num_tokens=model.num_tokens,
            row_hidden=LENS_MAX_ROW_HIDDEN,
            element_hidden=LENS_MAX_ELEMENT_HIDDEN,
        )


def checkpoint_model_state(checkpoint):
    return checkpoint["model"] if "model" in checkpoint else checkpoint


def make_plain_model():
    return CIFAR10SHViT(
        image_size=IMAGE_SIZE,
        patch_size=PATCH_SIZE,
        num_classes=NUM_CLASSES,
        dim=EMBED_DIM,
        depth=DEPTH,
        mlp_ratio=MLP_RATIO,
        drop=DROPOUT,
        attn_drop=ATTENTION_DROPOUT,
    )


def make_full_lens_model():
    model = make_plain_model()
    install_lens_norm(model)
    install_lens_max(model)
    return model


def load_state(path):
    checkpoint = torch.load(path, map_location="cpu")
    return checkpoint_model_state(checkpoint)


def save_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2), encoding="utf-8")


In [4]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])


def make_analysis_loaders(seed):
    validation_dataset = datasets.CIFAR10(
        root=DATA_ROOT,
        train=True,
        download=False,
        transform=eval_transform,
    )
    test_dataset = datasets.CIFAR10(
        root=DATA_ROOT,
        train=False,
        download=False,
        transform=eval_transform,
    )

    split_generator = torch.Generator().manual_seed(seed)
    indices = torch.randperm(len(validation_dataset), generator=split_generator).tolist()
    validation_indices = indices[:VALIDATION_SIZE]
    validation_subset = Subset(validation_dataset, validation_indices)
    pin_memory = DEVICE.type == "cuda"

    validation_loader = DataLoader(
        validation_subset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=pin_memory,
        persistent_workers=NUM_WORKERS > 0,
    )

    curve_generator = torch.Generator().manual_seed(seed)
    curve_indices = torch.randperm(len(test_dataset), generator=curve_generator)[:CURVE_BATCH_SIZE].tolist()
    curve_subset = Subset(test_dataset, curve_indices)
    curve_loader = DataLoader(
        curve_subset,
        batch_size=CURVE_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=pin_memory,
        persistent_workers=NUM_WORKERS > 0,
    )
    return validation_loader, curve_loader, curve_indices


@torch.no_grad()
def evaluate(model, data_loader):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for images, targets in data_loader:
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        logits = model(images)
        loss = F.cross_entropy(logits, targets)
        total_loss += loss.item() * targets.size(0)
        total_correct += (logits.argmax(dim=1) == targets).sum().item()
        total_examples += targets.size(0)

    return {
        "loss": total_loss / total_examples,
        "accuracy": total_correct / total_examples,
    }


validation_loader, curve_loader, curve_test_indices = make_analysis_loaders(SEED)
print(f"validation_examples={len(validation_loader.dataset)}")
print(f"curve_test_indices={curve_test_indices}")


validation_examples=5000
curve_test_indices=[7542, 8214, 3698, 2841, 8086, 2550, 2282, 7599, 1462, 2026, 2048, 1871, 6934, 6493, 3819, 717, 6771, 3217, 2519, 6359, 6929, 5756, 5897, 9764, 5307, 6113, 6903, 3841, 5051, 587, 6642, 9841]


## 1. Relative L2 of shared inherited parameters

For each matching tensor, the notebook computes

\[
\frac{\lVert \theta_{\mathrm{recovered}}-\theta_{\mathrm{baseline}}\rVert_2}
{\lVert \theta_{\mathrm{baseline}}\rVert_2+\epsilon}.
\]

The analysis is restricted to the four inherited linear transformations inside each Transformer block: QKV, QKV projection, FFN FC0, and FFN FC1. Weight and bias tensors are shown separately, and no tensor-size-weighted aggregation is performed.


In [5]:
class BoldLogFormatter(LogFormatterSciNotation):
    def __call__(self, value, position=None):
        label = super().__call__(value, position)
        return label.replace(r"\mathdefault{", r"\mathbf{")


baseline_state = load_state(BASELINE_BEST)
recovered_state = load_state(RECOVERED_LENS_BEST)

RELATIVE_L2_MODULES = {
    "attn.qkv": "QKV Proj.",
    "attn.proj": "Out. Proj.",
    "mlp.fc1": "FFN FC0",
    "mlp.fc2": "FFN FC1",
}

rows = []
for name in sorted(set(baseline_state) & set(recovered_state)):
    name_parts = name.split(".")
    if len(name_parts) != 5 or name_parts[0] != "blocks":
        continue
    module_name = ".".join(name_parts[2:-1])
    if module_name not in RELATIVE_L2_MODULES:
        continue

    baseline_tensor = baseline_state[name]
    recovered_tensor = recovered_state[name]
    if baseline_tensor.shape != recovered_tensor.shape:
        continue
    if not (torch.is_floating_point(baseline_tensor) and torch.is_floating_point(recovered_tensor)):
        continue
    if "lens_max" in name or "predictor_" in name:
        continue

    baseline_float = baseline_tensor.detach().float()
    recovered_float = recovered_tensor.detach().float()
    baseline_l2 = torch.linalg.vector_norm(baseline_float).item()
    delta_l2 = torch.linalg.vector_norm(recovered_float - baseline_float).item()
    parameter_type = "bias" if name.endswith(".bias") else "weight"
    rows.append({
        "parameter": name,
        "parameter_type": parameter_type,
        "numel": baseline_tensor.numel(),
        "baseline_l2": baseline_l2,
        "delta_l2": delta_l2,
        "relative_l2": delta_l2 / (baseline_l2 + EPS),
    })

csv_path = ANALYSIS_ROOT / "Relative_L2" / "relative_l2_selected_parameters.csv"
with csv_path.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)
save_json(ANALYSIS_ROOT / "Relative_L2" / "relative_l2_selected_parameters.json", rows)


HEATMAP_ROWS = ["Block 0", "Block 1", "Block 2"]
HEATMAP_COLUMNS = ["QKV Proj.", "Out. Proj.", "FFN FC0", "FFN FC1"]


def heatmap_location(parameter_name):
    parts = parameter_name.split(".")
    if len(parts) == 5 and parts[0] == "blocks":
        row_name = f"Block {int(parts[1])}"
        module_name = ".".join(parts[2:-1])
        if module_name in RELATIVE_L2_MODULES:
            return row_name, RELATIVE_L2_MODULES[module_name]
    raise ValueError(f"Unmapped shared parameter: {parameter_name}")


def plot_relative_l2_heatmap(selected_rows, columns, output_dir, title):
    matrix = np.full((len(HEATMAP_ROWS), len(columns)), np.nan, dtype=np.float64)
    row_lookup = {name: index for index, name in enumerate(HEATMAP_ROWS)}
    column_lookup = {name: index for index, name in enumerate(columns)}

    for item in selected_rows:
        row_name, column_name = heatmap_location(item["parameter"])
        if column_name not in column_lookup:
            raise ValueError(f"Unexpected column for this heatmap: {item['parameter']}")
        matrix[row_lookup[row_name], column_lookup[column_name]] = item["relative_l2"]

    csv_matrix_path = output_dir / "relative_l2_matrix.csv"
    with csv_matrix_path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["stage"] + columns)
        for row_name, values in zip(HEATMAP_ROWS, matrix):
            writer.writerow([row_name] + ["" if np.isnan(value) else value for value in values])

    finite_positive = matrix[np.isfinite(matrix) & (matrix > 0)]
    if finite_positive.size == 0:
        raise ValueError("Relative-L2 heatmap has no positive values")

    display_matrix = matrix.copy()
    smallest_positive = finite_positive.min()
    display_matrix[np.isfinite(display_matrix) & (display_matrix <= 0)] = smallest_positive
    color_min = finite_positive.min()
    color_max = finite_positive.max()
    color_norm = LogNorm(vmin=color_min, vmax=color_max) if color_max > color_min else None
    color_map = plt.colormaps["viridis"].copy()
    color_map.set_bad("#eeeeee")

    figure_height = 10.0
    row_height = figure_height * 0.72 / len(HEATMAP_ROWS)
    cell_font_size = max(7.0, min(12.0, row_height * 13.0))
    row_tick_font_size = max(8.0, min(13.0, row_height * 15.0))
    fixed_font_size = 20.0

    fig = plt.figure(figsize=(10.0, figure_height))
    ax = fig.add_axes([0.10, 0.13, 0.72, 0.76])
    colorbar_ax = fig.add_axes([0.85, 0.13, 0.035, 0.76])
    image = ax.imshow(np.ma.masked_invalid(display_matrix), cmap=color_map, norm=color_norm, aspect="auto")
    ax.set_xticks(np.arange(len(columns)), labels=columns)
    ax.set_yticks(np.arange(len(HEATMAP_ROWS)), labels=HEATMAP_ROWS)
    for tick_label in ax.get_xticklabels():
        tick_label.set_fontweight("bold")
        tick_label.set_fontsize(15)
    for tick_label in ax.get_yticklabels():
        tick_label.set_fontweight("bold")
        tick_label.set_fontsize(row_tick_font_size)
    # ax.set_title(
    #     f"{title}\nvmin={color_min:.2e}, vmax={color_max:.2e}",
    #     fontsize=fixed_font_size,
    #     fontweight="bold",
    # )
    ax.set_title(
        f"vmin={color_min:.2e}, vmax={color_max:.2e}",
        fontsize=fixed_font_size,
        fontweight="bold",
    )

    for row_index in range(matrix.shape[0]):
        for column_index in range(matrix.shape[1]):
            value = matrix[row_index, column_index]
            if np.isfinite(value):
                ax.text(
                    column_index, row_index, f"{value:.2e}",
                    ha="center", va="center", fontsize=cell_font_size,
                    fontweight="bold", color="white",
                )

    colorbar = fig.colorbar(image, cax=colorbar_ax)
    colorbar.formatter = BoldLogFormatter()
    colorbar.update_ticks()
    colorbar.set_label(
        "Relative L2 (logarithmic color scale)",
        fontsize=fixed_font_size,
        fontweight="bold",
    )
    colorbar.ax.tick_params(labelsize=fixed_font_size)
    for tick_label in colorbar.ax.get_yticklabels():
        tick_label.set_fontweight("bold")
    fig.savefig(output_dir / "relative_l2_heatmap.pdf", dpi=300)
    plt.close(fig)
    save_json(output_dir / "relative_l2.json", selected_rows)


weight_rows = [row for row in rows if row["parameter_type"] == "weight"]
bias_rows = [row for row in rows if row["parameter_type"] == "bias"]
plot_relative_l2_heatmap(
    weight_rows, HEATMAP_COLUMNS, CATEGORY_DIRS["relative_weights"],
    "Relative L2 of shared inherited weights after LENS recovery",
)
plot_relative_l2_heatmap(
    bias_rows, HEATMAP_COLUMNS, CATEGORY_DIRS["relative_biases"],
    "Relative L2 of shared inherited biases after LENS recovery",
)

print(f"shared tensors={len(rows)} | weights={len(weight_rows)} | biases={len(bias_rows)}")
print(f"saved={csv_path}")


shared tensors=24 | weights=12 | biases=12
saved=analysis/Relative_L2/relative_l2_selected_parameters.csv


## 2. Matched test-set function curves

For every operator module, one figure overlays 256 randomly selected test-set curves. LayerNorm/LENSNorm curves are identified by `(module, image, token)`, while Softmax/LENSMax curves are identified by `(attention module, image, query)`. Corresponding baseline and LENS figures use exactly the same image and location selections.


In [6]:
def load_analysis_models():
    baseline_model = make_plain_model()
    baseline_model.load_state_dict(baseline_state, strict=True)

    lens_model = make_full_lens_model()
    lens_model.load_state_dict(recovered_state, strict=True)

    baseline_model = baseline_model.to(DEVICE).eval()
    lens_model = lens_model.to(DEVICE).eval()
    return baseline_model, lens_model


@torch.no_grad()
def collect_baseline_activations(model, images):
    norm_values = {}
    softmax_values = {}
    handles = []

    for name, module in model.named_modules():
        if isinstance(module, nn.LayerNorm):
            def norm_hook(current_name):
                def hook(_module, inputs, output):
                    norm_values[current_name] = (
                        inputs[0].detach().cpu(),
                        output.detach().cpu(),
                    )
                return hook
            handles.append(module.register_forward_hook(norm_hook(name)))

        if isinstance(module, SingleHeadSelfAttention):
            def attention_hook(current_name):
                def hook(current_module, inputs, _output):
                    x = inputs[0]
                    batch_size, num_tokens, dim = x.shape
                    qkv = current_module.qkv(x).reshape(batch_size, num_tokens, 3, dim).permute(2, 0, 1, 3)
                    q, k, _v = qkv.unbind(0)
                    scores = (q @ k.transpose(-2, -1)) * current_module.scale
                    probabilities = F.softmax(scores, dim=-1)
                    softmax_values[current_name] = (
                        scores.detach().cpu(),
                        probabilities.detach().cpu(),
                    )
                return hook
            handles.append(module.register_forward_hook(attention_hook(name)))

    model(images)
    for handle in handles:
        handle.remove()
    return norm_values, softmax_values


@torch.no_grad()
def collect_lens_activations(model, images):
    norm_values = {}
    max_values = {}
    handles = []

    for name, module in model.named_modules():
        if isinstance(module, LENSNorm):
            def norm_hook(current_name):
                def hook(_module, inputs, output):
                    norm_values[current_name] = (
                        inputs[0].detach().cpu(),
                        output.detach().cpu(),
                    )
                return hook
            handles.append(module.register_forward_hook(norm_hook(name)))

        if isinstance(module, LENSMax):
            parent_attention_name = name.removesuffix(".lens_max")
            def max_hook(current_name):
                def hook(_module, inputs, output):
                    max_values[current_name] = (
                        inputs[0].detach().cpu(),
                        output.detach().cpu(),
                    )
                return hook
            handles.append(module.register_forward_hook(max_hook(parent_attention_name)))

    model(images)
    for handle in handles:
        handle.remove()
    return norm_values, max_values


def module_directory_name(module_name):
    return module_name.replace(".", "_")


def save_curve_set(category, module_name, curves, metadata, title, x_label, y_label):
    module_dir = CATEGORY_DIRS[category] / module_directory_name(module_name)
    module_dir.mkdir(parents=True, exist_ok=True)

    arrays = {}
    fig, ax = plt.subplots(figsize=(7.2, 5.2))
    for curve_index, (x, y) in enumerate(curves, start=1):
        order = np.argsort(x)
        sorted_x = np.asarray(x)[order]
        sorted_y = np.asarray(y)[order]
        arrays[f"x_{curve_index:02d}"] = sorted_x
        arrays[f"y_{curve_index:02d}"] = sorted_y
        ax.plot(
            sorted_x,
            sorted_y,
            linewidth=2.0,
            alpha=0.20,
            color="#4472C4",
        )

    np.savez(module_dir / "curve_data.npz", **arrays)
    save_json(module_dir / "metadata.json", metadata)
    # ax.set_xlabel(x_label)
    # ax.set_ylabel(y_label)
    # ax.set_title(title)

    ax.set_title(
    title,
    fontsize=20,
    fontweight="bold",
    pad=10,
    )
    
    ax.set_xlabel(
        x_label,
        fontsize=18,
        fontweight="bold",
    )
    
    ax.set_ylabel(
        y_label,
        fontsize=18,
        fontweight="bold",
    )
    
    ax.tick_params(
        axis="both",
        labelsize=14,
        width=1.5,
    )
    
    for tick_label in ax.get_xticklabels() + ax.get_yticklabels():
        tick_label.set_fontweight("bold")
    
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(module_dir / "curves.pdf", dpi=220, bbox_inches="tight")
    plt.close(fig)


baseline_model, lens_model = load_analysis_models()
curve_images, _curve_targets = next(iter(curve_loader))
curve_images = curve_images.to(DEVICE, non_blocking=True)

baseline_norm, baseline_softmax = collect_baseline_activations(baseline_model, curve_images)
lens_norm, lens_max = collect_lens_activations(lens_model, curve_images)

common_norm_names = sorted(set(baseline_norm) & set(lens_norm))
common_attention_names = sorted(set(baseline_softmax) & set(lens_max))
if not common_norm_names or not common_attention_names:
    raise RuntimeError("Could not find matched baseline/LENS modules")

rng = np.random.default_rng(SEED)
selection_manifest = {"LayerNorm_LENSNorm": {}, "Softmax_LENSMax": {}}

for module_name in common_norm_names:
    values = baseline_norm[module_name][0]
    candidates = [
        (batch_index, token_index)
        for batch_index in range(values.shape[0])
        for token_index in range(values.shape[1])
    ]
    selected_indices = rng.choice(len(candidates), size=CURVES_PER_MODULE, replace=False)
    selected_locations = [candidates[int(index)] for index in selected_indices]
    baseline_curves = []
    lens_curves = []
    instances = []

    for curve_number, (batch_index, token_index) in enumerate(selected_locations, start=1):
        baseline_curves.append((
            baseline_norm[module_name][0][batch_index, token_index].numpy(),
            baseline_norm[module_name][1][batch_index, token_index].numpy(),
        ))
        lens_curves.append((
            lens_norm[module_name][0][batch_index, token_index].numpy(),
            lens_norm[module_name][1][batch_index, token_index].numpy(),
        ))
        instances.append({
            "curve": curve_number,
            "batch_index": batch_index,
            "test_dataset_index": curve_test_indices[batch_index],
            "token_index": token_index,
        })

    selection_manifest["LayerNorm_LENSNorm"][module_name] = instances
    save_curve_set(
        "layernorm", module_name, baseline_curves,
        {"operator": "LayerNorm", "module": module_name, "checkpoint": str(BASELINE_BEST), "instances": instances},
        f"LayerNorm curves — {module_name}", "Input activation", "Output activation",
    )
    save_curve_set(
        "lensnorm", module_name, lens_curves,
        {"operator": "LENSNorm", "module": module_name, "checkpoint": str(RECOVERED_LENS_BEST), "instances": instances},
        f"LENSNorm curves — {module_name}", "Input activation", "Output activation",
    )

for module_name in common_attention_names:
    values = baseline_softmax[module_name][0]
    candidates = [
        (batch_index, query_index)
        for batch_index in range(values.shape[0])
        for query_index in range(values.shape[1])
    ]
    selected_indices = rng.choice(len(candidates), size=CURVES_PER_MODULE, replace=False)
    selected_locations = [candidates[int(index)] for index in selected_indices]
    baseline_curves = []
    lens_curves = []
    instances = []

    for curve_number, (batch_index, query_index) in enumerate(selected_locations, start=1):
        baseline_curves.append((
            baseline_softmax[module_name][0][batch_index, query_index].numpy(),
            baseline_softmax[module_name][1][batch_index, query_index].numpy(),
        ))
        lens_curves.append((
            lens_max[module_name][0][batch_index, query_index].numpy(),
            lens_max[module_name][1][batch_index, query_index].numpy(),
        ))
        instances.append({
            "curve": curve_number,
            "batch_index": batch_index,
            "test_dataset_index": curve_test_indices[batch_index],
            "query_index": query_index,
        })

    selection_manifest["Softmax_LENSMax"][module_name] = instances
    save_curve_set(
        "softmax", module_name, baseline_curves,
        {"operator": "Softmax", "module": module_name, "checkpoint": str(BASELINE_BEST), "instances": instances},
        f"Softmax curves — {module_name}", "Attention score", "Attention weight",
    )
    save_curve_set(
        "lensmax", module_name, lens_curves,
        {"operator": "LENSMax", "module": module_name, "checkpoint": str(RECOVERED_LENS_BEST), "instances": instances},
        f"LENSMax curves — {module_name}", "Attention score", "Attention weight",
    )

save_json(ANALYSIS_ROOT / "function_curve_selection_manifest.json", selection_manifest)

del baseline_model, lens_model, curve_images
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"saved {CURVES_PER_MODULE} curves for each of {len(common_norm_names)} LayerNorm/LENSNorm modules")
print(f"saved {CURVES_PER_MODULE} curves for each of {len(common_attention_names)} Softmax/LENSMax modules")


saved 256 curves for each of 7 LayerNorm/LENSNorm modules
saved 256 curves for each of 3 Softmax/LENSMax modules


## 3. Full training histories with normalized training progress

The four curves are:

1. baseline from scratch;
2. full LENS from scratch;
3. full LENS recovery with all inherited weights trainable;
4. full LENS recovery with inherited baseline weights frozen.

The baseline and full-LENS-from-scratch experiments use all 400 epochs, while the two recovery experiments use all 50 epochs. The horizontal axis is normalized to training progress from 0% to 100%, so complete trajectories with different budgets remain directly comparable.

Epoch 0 is evaluated from the exact initialization implied by each experiment. Experiments 7 and 8 have the same initial forward function; freezing parameters changes optimization, not the epoch-0 output.


In [7]:
def evaluate_epoch0(kind):
    seed_everything(SEED)

    if kind == "baseline":
        model = make_plain_model()
    elif kind == "scratch_lens":
        model = make_full_lens_model()
    elif kind in {"recovered_lens", "frozen_lens"}:
        model = make_plain_model()
        model.load_state_dict(baseline_state, strict=True)
        install_lens_norm(model)
        install_lens_max(model)
        if kind == "frozen_lens":
            for parameter in model.parameters():
                parameter.requires_grad = False
            for module in model.modules():
                if isinstance(module, (LENSNorm, LENSMax)):
                    for parameter in module.parameters():
                        parameter.requires_grad = True
    else:
        raise ValueError(f"Unknown initialization: {kind}")

    model = model.to(DEVICE)
    metrics = evaluate(model, validation_loader)
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return metrics


history_specs = [
    {
        "key": "baseline",
        "label": "Baseline from scratch",
        "color": "#1f77b4",
        "directory": BASELINE_DIR,
        "total_epochs": 400,
    },
    {
        "key": "scratch_lens",
        "label": "Full LENS from scratch",
        "color": "#ff7f0e",
        "directory": SCRATCH_LENS_DIR,
        "total_epochs": 400,
    },
    {
        "key": "recovered_lens",
        "label": "Full LENS recovery (all weights)",
        "color": "#2ca02c",
        "directory": RECOVERED_LENS_DIR,
        "total_epochs": 50,
    },
    {
        "key": "frozen_lens",
        "label": "Full LENS recovery (baseline frozen)",
        "color": "#d62728",
        "directory": FROZEN_LENS_DIR,
        "total_epochs": 50,
    },
]

epoch0_metrics = {}
history_data = {}
for spec in history_specs:
    epoch0_metrics[spec["key"]] = evaluate_epoch0(spec["key"])
    records = json.loads((spec["directory"] / "history.json").read_text(encoding="utf-8"))
    records = sorted(records, key=lambda item: item["epoch"])
    records = [record for record in records if 1 <= int(record["epoch"]) <= spec["total_epochs"]]
    if len(records) != spec["total_epochs"]:
        raise ValueError(
            f"{spec['directory'] / 'history.json'} contains {len(records)} records "
            f"for epochs 1..{spec['total_epochs']}, expected {spec['total_epochs']}"
        )
    history_data[spec["key"]] = records

save_json(CATEGORY_DIRS["history"] / "epoch0_validation.json", epoch0_metrics)
save_json(
    CATEGORY_DIRS["history"] / "full_histories_with_epoch0.json",
    {
        spec["key"]: {
            "label": spec["label"],
            "total_epochs": spec["total_epochs"],
            "epoch0_validation": epoch0_metrics[spec["key"]],
            "history": history_data[spec["key"]],
        }
        for spec in history_specs
    },
)


def plot_history(metric, title, y_label, include_epoch0, filename):
    fig, ax = plt.subplots(figsize=(8.4, 5.4))
    for spec in history_specs:
        records = history_data[spec["key"]]
        epochs = [int(record["epoch"]) for record in records]
        progress = [100.0 * epoch / spec["total_epochs"] for epoch in epochs]
        values = [float(record[metric]) for record in records]
        if include_epoch0:
            epoch0_name = "accuracy" if metric == "validation_accuracy" else "loss"
            progress = [0.0] + progress
            values = [float(epoch0_metrics[spec["key"]][epoch0_name])] + values
        ax.plot(progress, values, linewidth=3.5, color=spec["color"], label=spec["label"])

    # ax.set_xlabel("Training progress (%)")
    # ax.set_ylabel(y_label)
    # ax.set_title(title)
    
    ax.set_title(
    title,
    fontsize=20,
    fontweight="bold",
    pad=10,
    )
    
    ax.set_xlabel(
        "Training progress (%)",
        fontsize=18,
        fontweight="bold",
    )
    
    ax.set_ylabel(
        y_label,
        fontsize=18,
        fontweight="bold",
    )

    ax.tick_params(
        axis="both",
        labelsize=14,
        width=1.5,
    )
    
    for tick_label in ax.get_xticklabels() + ax.get_yticklabels():
        tick_label.set_fontweight("bold")
    
    ax.set_xlim(0, 100)
    ax.grid(alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(CATEGORY_DIRS["history"] / filename, dpi=220, bbox_inches="tight")
    plt.close(fig)


plot_history(
    "train_accuracy", "CIFAR-10 Training Top-1 Accuracy on SHViT", "Accuracy", False,
    "training_accuracy.pdf",
)
plot_history(
    "validation_accuracy", "CIFAR-10 Validation Top-1 Accuracy on SHViT", "Accuracy", True,
    "validation_accuracy.pdf",
)
plot_history(
    "train_loss", "CIFAR-10 Training Loss", "Cross-entropy loss on SHViT", False,
    "training_loss.pdf",
)
plot_history(
    "validation_loss", "CIFAR-10 Validation Loss", "Cross-entropy loss on SHViT", True,
    "validation_loss.pdf",
)

print(json.dumps(epoch0_metrics, indent=2))
print(f"saved training-history outputs to {CATEGORY_DIRS['history']}")


{
  "baseline": {
    "loss": 2.3398575004577635,
    "accuracy": 0.0748
  },
  "scratch_lens": {
    "loss": 2.302497632598877,
    "accuracy": 0.1304
  },
  "recovered_lens": {
    "loss": 6.815776066589356,
    "accuracy": 0.156
  },
  "frozen_lens": {
    "loss": 6.815776066589356,
    "accuracy": 0.156
  }
}
saved training-history outputs to analysis/Training_History


In [8]:
print("Analysis complete.")
for name, directory in CATEGORY_DIRS.items():
    print(f"{name}: {directory.resolve()}")


Analysis complete.
relative_weights: /home/yuhao/project/LENS/SHViT/CIFAR10/analysis/Relative_L2/weights
relative_biases: /home/yuhao/project/LENS/SHViT/CIFAR10/analysis/Relative_L2/biases
layernorm: /home/yuhao/project/LENS/SHViT/CIFAR10/analysis/LayerNorm
softmax: /home/yuhao/project/LENS/SHViT/CIFAR10/analysis/Softmax
lensnorm: /home/yuhao/project/LENS/SHViT/CIFAR10/analysis/LENSNorm
lensmax: /home/yuhao/project/LENS/SHViT/CIFAR10/analysis/LENSMax
history: /home/yuhao/project/LENS/SHViT/CIFAR10/analysis/Training_History
